# Scenario A · RF 第二阶段抽样

使用桌面服务器快照中已经拟合的 RF 缓存。每个有数据的条件不放回抽 200 次。当前有 20 组，共 4,000 次。其中 n=2,000 的 Logistic 从已保存的 778 条中抽 200 条。

请选本机 Python 3 (environment2) 或装好备份包 requirements.txt 中版本的内核。设 START 为 True 后运行下面的代码格。重新运行会跳过已保存的记录。

In [ ]:
from pathlib import Path
import os
import signal
import subprocess
import sys

import psutil
from IPython.display import clear_output

START = False
WORKERS = 12

desktop = Path.home() / "Desktop" / "Rank-DML_RF_服务器备份_20261123_000221"
fallback = desktop / "RF_二阶段抽样200" / "RF实验_代码"
folders = (Path.cwd(), *Path.cwd().parents, fallback)
script = next((p / "run_stage2_sample.py" for p in folders if (p / "run_stage2_sample.py").is_file()), None)

if not START:
    print("请把 START 改成 True 后运行本格。")
elif script is None:
    raise FileNotFoundError("请从 RF_二阶段抽样200 项目中打开此 Notebook。")
else:
    on_windows = os.name == "nt"
    env = os.environ | {"PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1"}
    with subprocess.Popen(
        [sys.executable, "-B", "-u", str(script), "--workers", str(WORKERS)],
        cwd=script.parent, env=env, text=True, encoding="utf-8", errors="replace",
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=not on_windows,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0) if on_windows else 0,
    ) as process:
        try:
            for line in process.stdout:
                if "RF Stage 2:" in line:
                    clear_output(wait=True)
                if line.strip():
                    print(line.strip(), flush=True)
            code = process.wait()
            if code:
                raise RuntimeError(f"程序退出码 {code}。请查看本格的错误输出。")
        except KeyboardInterrupt:
            if process.poll() is None:
                if on_windows:
                    owner = psutil.Process(process.pid)
                    children = owner.children(recursive=True)
                    for item in [owner, *children]:
                        try:
                            item.terminate()
                        except psutil.NoSuchProcess:
                            pass
                    psutil.wait_procs(children, timeout=5)
                else:
                    try:
                        os.killpg(process.pid, signal.SIGTERM)
                    except ProcessLookupError:
                        pass
                process.wait()
            print("已停止。重新运行本格会从已保存的抽样结果续跑。")